# 17. Combined FT без synthetic replay

Этот ноутбук сохраняет проверенную реализацию исходного эксперимента и имена его
артефактов. Запускайте **Restart Kernel and Run All Cells** после выполнения всех
предыдущих пронумерованных ноутбуков.

Все входы, кроме исходных raw-данных из `config/raw_sources.json`, создаются внутри
этого проекта. Результаты записываются в `outputs/`, а модели — в `checkpoints/`.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Неверный путь.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from project_paths import *
ensure_project_directories()

print("Project root:", PROJECT_ROOT)

Project root: D:\Users\user\Desktop\DS_XRD_project


# Контрольный FT: RRUFF + opXRD без synthetic replay

Этот ноутбук нужен для прямой проверки гипотезы: мешает ли совместное fine-tuning на двух разных реальных источниках.

Он отличается от двух source-only ноутбуков только составом train:

- стартует с того же synthetic pretrain V2;
- fine-tuning использует RRUFF и opXRD вместе;
- synthetic replay полностью отключён;
- learning rate, weight decay, число эпох и порог elements подбираются внутри train;
- итоговые метрики считаются отдельно на RRUFF, отдельно на opXRD и на их объединении.

Внешние folds строятся отдельно для каждого источника тем же GroupKFold, что и в source-only опытах, а затем объединяются. Поэтому сравнение со специалистами является попарным: модели проверяются на одинаковых строках каждого источника.

In [2]:
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = DEVICE.type == 'cuda'
device_name = torch.cuda.get_device_name(0) if AMP else 'CPU'
print('device:', DEVICE, '|', device_name)

device: cuda | NVIDIA GeForce RTX 3060 Ti


In [3]:
# ---------- настройки контрольного эксперимента ----------
MODE = 'cv'  # 'cv' — полный nested 5-fold; 'quick' — один fold для проверки

SOURCE_NAME = 'RRUFF + opXRD'
SOURCE_KEY = 'combined'
OUTPUT_PREFIX = 'ft_combined_no_replay_control'

BASE = PROJECT_ROOT
DATA = BASE / 'data' / 'preprocessed'
CKPT_PRE = BASE / 'checkpoints' / 'pretrain_v2_full_best.pt'
OUT = BASE / 'outputs'
CKPT_DIR = BASE / 'checkpoints'

GRID_N = 4096
W = 3
BATCH = 64
OUTER_FOLDS = 5
INNER_VAL_FRAC = 0.20
MAX_TUNE_EPOCHS = 30 if MODE == 'cv' else 8
PATIENCE = 7 if MODE == 'cv' else 3
MIN_EPOCHS = 5 if MODE == 'cv' else 2
WARMUP_FRAC = 0.10
CLIP = 1.0
EL_THRESHOLDS = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]

SYSTEMS = ['triclinic', 'monoclinic', 'orthorhombic', 'tetragonal',
           'trigonal', 'hexagonal', 'cubic']
IMPUTE_LAMBDA = 1.5406
W_LAT, W_VOL, W_SG, W_SYS, W_EL, W_ANG = 1.0, 0.5, 1.0, 1.0, 1.0, 2.0

# Сетка содержит точные финальные режимы обоих специалистов и два промежуточных.
CANDIDATES = [
    {'name': 'head_only', 'lr_backbone': 0.0, 'lr_head': 8e-5, 'wd': 1e-4},
    {'name': 'conservative', 'lr_backbone': 3e-6, 'lr_head': 3e-5, 'wd': 1e-4},
    {'name': 'balanced', 'lr_backbone': 8e-6, 'lr_head': 5e-5, 'wd': 1e-4},
    {'name': 'stronger', 'lr_backbone': 1.5e-5, 'lr_head': 1e-4, 'wd': 3e-4},
]

print('mode:', MODE)
print('candidates:', [c['name'] for c in CANDIDATES])
print('fine-tuning: RRUFF + opXRD | synthetic replay: OFF')

mode: cv
candidates: ['head_only', 'conservative', 'balanced', 'stronger']
fine-tuning: RRUFF + opXRD | synthetic replay: OFF


In [4]:
# ---------- два real-пула, источник сохраняется явно ----------
stats = json.loads((OUT / 'pretrain_stats.json').read_text())
VOCAB = stats['vocab']
EL_IDX = {e: i for i, e in enumerate(VOCAB)}
LAT_MEAN = np.array(stats['lat_mean'])
LAT_STD = np.array(stats['lat_std'])
VOL_MEAN, VOL_STD = stats['vol_mean'], stats['vol_std']

index = pd.read_parquet(DATA / 'index_preprocessed.parquet')
N_TOTAL = len(index)
X_MM = np.memmap(DATA / 'X_intensity.f16', dtype=np.float16, mode='r',
                 shape=(N_TOTAL, GRID_N))
M_MM = np.memmap(DATA / 'M_mask.u8', dtype=np.uint8, mode='r',
                 shape=(N_TOTAL, GRID_N))
row_of = dict(zip(index['sample_id'], index['row_idx']))

def to_list(v):
    if isinstance(v, str):
        try:
            parsed = json.loads(v)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []
    if isinstance(v, (list, tuple, np.ndarray)):
        return [e for e in v if isinstance(e, str)]
    return []

def prepare_pool(filename, source_name):
    frame = pd.read_parquet(BASE / 'data' / 'clean' / filename).copy()
    frame['dataset_source'] = source_name
    frame['row_idx'] = frame['sample_id'].map(row_of)
    assert frame['row_idx'].notna().all()
    frame['primary_wavelength'] = frame['primary_wavelength'].fillna(IMPUTE_LAMBDA)
    if 'secondary_wavelength' not in frame.columns:
        frame['secondary_wavelength'] = np.nan
    frame['crystal_system'] = frame['crystal_system'].replace({'rhombohedral': 'trigonal'})

    abc = frame[['lattice_a', 'lattice_b', 'lattice_c']].to_numpy(np.float64)
    ang = frame[['alpha', 'beta', 'gamma']].to_numpy(np.float64)
    ca, cb, cg = (np.cos(np.radians(ang[:, i])) for i in range(3))
    volume_term = 1.0 - ca**2 - cb**2 - cg**2 + 2 * ca * cb * cg
    frame['V'] = abc.prod(1) * np.sqrt(np.clip(volume_term, 1e-12, None))
    frame['lat6'] = list(np.hstack([np.log(abc), ang]))
    frame['elements'] = frame['elements_list'].apply(to_list)
    composition = frame['phase_compositions'].fillna('').astype(str)
    frame['conn_key'] = composition + '|' + frame['lattice_a'].round(2).astype(str)
    return frame.reset_index(drop=True)

ft_rruff = prepare_pool('ft_pool_rruff.parquet', 'rruff')
ft_opxrd = prepare_pool('ft_pool_opxrd.parquet', 'opxrd')
ft = pd.concat([ft_rruff, ft_opxrd], ignore_index=True)

def coverage(frame):
    return dict(
        rows=len(frame), groups=frame['conn_key'].nunique(),
        lattice=int(frame['lattice_a'].notna().sum()),
        system=int(frame['crystal_system'].isin(SYSTEMS).sum()),
        space_group=int(frame['spacegroup_number'].notna().sum()),
        elements=int(frame['elements'].map(len).gt(0).sum()),
    )

print('RRUFF:', coverage(ft_rruff))
print('opXRD:', coverage(ft_opxrd))
print('combined rows:', len(ft))
assert len(ft_rruff) == 1359 and len(ft_opxrd) == 2116

RRUFF: {'rows': 1359, 'groups': 1251, 'lattice': 1298, 'system': 1298, 'space_group': 0, 'elements': 1357}
opXRD: {'rows': 2116, 'groups': 778, 'lattice': 1355, 'system': 1355, 'space_group': 570, 'elements': 896}
combined rows: 3475


In [5]:
# ---------- Dataset и маски неполных меток ----------
def build_labels(frame):
    lam1 = frame['primary_wavelength'].to_numpy(np.float32)
    lam2 = frame['secondary_wavelength'].to_numpy(np.float32)
    lam = np.stack([lam1 / 1.54, np.nan_to_num(lam2) / 1.54,
                    np.isfinite(lam2).astype(np.float32)], axis=1)

    lat6 = np.stack(frame['lat6'].to_numpy())
    has_lat = ~np.isnan(lat6).any(axis=1)
    latm = has_lat.astype(np.float32)
    lat6 = (lat6 - LAT_MEAN) / LAT_STD

    vol_raw = frame['V'].to_numpy(np.float64)
    vol = (np.log(vol_raw) - VOL_MEAN) / VOL_STD

    sg_raw = frame['spacegroup_number'].to_numpy(float)
    sgm = np.isfinite(sg_raw).astype(np.float32)
    sg = np.nan_to_num(sg_raw, nan=1.0).astype(np.int64) - 1

    sysmap = {s: i for i, s in enumerate(SYSTEMS)}
    sys_raw = frame['crystal_system'].map(sysmap)
    sysm = sys_raw.notna().to_numpy(np.float32)
    sys_ = sys_raw.fillna(0).to_numpy(np.int64)

    n = len(frame)
    el = np.zeros((n, len(VOCAB)), np.float32)
    elm = np.zeros(n, np.float32)
    for i, els in enumerate(frame['elements']):
        if els:
            elm[i] = 1.0
            for e in els:
                j = EL_IDX.get(e)
                if j is not None:
                    el[i, j] = 1.0

    return dict(lam=lam, lat6=lat6.astype(np.float32), latm=latm,
                vol=vol.astype(np.float32), volm=latm.copy(),
                sg=sg, sgm=sgm, sys_=sys_, sysm=sysm, el=el, elm=elm)

class SpecDS(Dataset):
    def __init__(self, frame):
        self.row = frame['row_idx'].to_numpy(np.int64)
        self.L = build_labels(frame)

    def __len__(self):
        return len(self.row)

    def __getitem__(self, i):
        x = np.empty((2, GRID_N), np.float32)
        x[0] = X_MM[self.row[i]]
        x[1] = M_MM[self.row[i]]
        L = self.L
        return (torch.from_numpy(x), torch.from_numpy(L['lam'][i]),
                torch.from_numpy(L['lat6'][i]), torch.tensor(L['latm'][i]),
                torch.tensor(L['sg'][i]), torch.tensor(L['sgm'][i]),
                torch.tensor(L['sys_'][i]), torch.tensor(L['sysm'][i]),
                torch.from_numpy(L['el'][i]), torch.tensor(L['elm'][i]),
                torch.tensor(L['vol'][i]), torch.tensor(L['volm'][i]))

def make_loader(frame, shuffle):
    return DataLoader(SpecDS(frame), batch_size=BATCH, shuffle=shuffle,
                      pin_memory=AMP, num_workers=0, drop_last=False)

In [6]:
# ---------- модель v2 ----------
class ResBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(cin, cout, 3, stride=stride, padding=1, bias=False)
        self.n1 = nn.GroupNorm(8, cout)
        self.conv2 = nn.Conv1d(cout, cout, 3, padding=1, bias=False)
        self.n2 = nn.GroupNorm(8, cout)
        if cin == cout and stride == 1:
            self.skip = nn.Identity()
        else:
            self.skip = nn.Sequential(nn.Conv1d(cin, cout, 1, stride=stride, bias=False),
                                      nn.GroupNorm(8, cout))

    def forward(self, x):
        h = F.gelu(self.n1(self.conv1(x)))
        h = self.n2(self.conv2(h))
        return F.gelu(h + self.skip(x))

class XRDNetV2(nn.Module):
    def __init__(self, n_el, w=3):
        super().__init__()
        c = [32 * w, 48 * w, 64 * w, 96 * w, 128 * w, 192 * w, 256 * w]
        self.stem = nn.Sequential(nn.Conv1d(2, c[0], 15, padding=7, bias=False),
                                  nn.GroupNorm(8, c[0]), nn.GELU())
        self.blocks = nn.Sequential(*[ResBlock(c[i], c[i + 1], stride=2)
                                      for i in range(6)])
        self.lam_mlp = nn.Sequential(nn.Linear(3, 16 * w), nn.GELU(), nn.Linear(16 * w, 16 * w))
        self.trunk = nn.Sequential(nn.Linear(c[-1] + 16 * w, 512 * w), nn.GELU(),
                                   nn.Linear(512 * w, 512 * w), nn.GELU())
        self.head_lat = nn.Linear(512 * w, 6)
        self.head_vol = nn.Linear(512 * w, 1)
        self.head_sg = nn.Linear(512 * w, 230)
        self.head_sys = nn.Linear(512 * w, 7)
        self.head_el = nn.Linear(512 * w, n_el)

    def forward(self, x, lam):
        f = self.stem(x)
        f = self.blocks(f)
        w = F.adaptive_avg_pool1d(x[:, 1:2], f.shape[-1]).clamp_min(1e-3)
        pooled = (f * w).sum(-1) / w.sum(-1)
        z = torch.cat([pooled, self.lam_mlp(lam)], dim=1)
        z = self.trunk(z)
        return dict(lat=self.head_lat(z), vol=self.head_vol(z).squeeze(-1),
                    sg=self.head_sg(z), sys=self.head_sys(z), el=self.head_el(z))

def load_pretrained():
    m = XRDNetV2(len(VOCAB), w=W).to(DEVICE)
    sd = torch.load(CKPT_PRE, map_location=DEVICE, weights_only=True)
    m.load_state_dict(sd)
    return m

print('модель v2 готова к загрузке чекпойнта')

модель v2 готова к загрузке чекпойнта


In [7]:
# ---------- Loss, все метрики и критерий подбора ----------
def masked_l1(pred, target, mask):
    m = mask > 0
    if m.sum() == 0:
        return pred.new_zeros(())
    return F.smooth_l1_loss(pred[m], target[m])

def masked_ce(logits, target, mask):
    m = mask > 0
    if m.sum() == 0:
        return logits.new_zeros(())
    return F.cross_entropy(logits[m], target[m].long())

def compute_losses(out, batch):
    (_, _, lat, latm, sg, sgm, sys_, sysm, el, elm, vol, volm) = batch
    m = latm > 0
    if m.sum() > 0:
        loss_len = F.smooth_l1_loss(out['lat'][m][:, :3], lat[m][:, :3])
        loss_ang = F.smooth_l1_loss(out['lat'][m][:, 3:], lat[m][:, 3:])
    else:
        loss_len = loss_ang = out['lat'].new_zeros(())

    me = elm > 0
    loss_el = (F.binary_cross_entropy_with_logits(out['el'][me], el[me])
               if me.sum() > 0 else out['el'].new_zeros(()))
    loss_vol = masked_l1(out['vol'], vol, volm)
    loss_sg = masked_ce(out['sg'], sg, sgm)
    loss_sys = masked_ce(out['sys'], sys_, sysm)
    total = (W_LAT * loss_len + W_ANG * loss_ang + W_VOL * loss_vol
             + W_SG * loss_sg + W_SYS * loss_sys + W_EL * loss_el)
    return total

@torch.no_grad()
def evaluate(model, loader, el_threshold=0.5):
    model.eval()
    rows = []
    for batch in loader:
        batch = [b.to(DEVICE, non_blocking=AMP) for b in batch]
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
            out = model(batch[0], batch[1])
        _, _, lat, latm, sg, sgm, sys_, sysm, el, elm, vol, volm = batch

        lat_pred = out['lat'].float().cpu().numpy() * LAT_STD + LAT_MEAN
        lat_true = lat.float().cpu().numpy() * LAT_STD + LAT_MEAN
        lat_pred[:, :3] = np.exp(lat_pred[:, :3])
        lat_true[:, :3] = np.exp(lat_true[:, :3])
        vol_pred = np.exp(out['vol'].float().cpu().numpy() * VOL_STD + VOL_MEAN)
        vol_true = np.exp(vol.float().cpu().numpy() * VOL_STD + VOL_MEAN)
        el_prob = torch.sigmoid(out['el']).float().cpu().numpy()
        el_true = el.float().cpu().numpy()

        for i in range(len(lat)):
            has_lat = float(latm[i]) > 0
            pred_set = frozenset(np.where(el_prob[i] > el_threshold)[0])
            true_set = (frozenset(np.where(el_true[i] > 0.5)[0])
                        if float(elm[i]) > 0 else frozenset())
            err_len = np.abs(lat_pred[i, :3] - lat_true[i, :3]) if has_lat else [np.nan] * 3
            ape_len = (100 * np.asarray(err_len) / np.maximum(lat_true[i, :3], 1e-6)
                       if has_lat else [np.nan] * 3)
            rows.append(dict(
                latm=float(latm[i]), sgm=float(sgm[i]), sysm=float(sysm[i]), elm=float(elm[i]),
                sg_ok=float(float(sgm[i]) > 0 and out['sg'][i].argmax().item() == sg[i].item()),
                sg_top5=float(float(sgm[i]) > 0 and sg[i].item() in out['sg'][i].topk(5).indices.tolist()),
                sys_ok=float(float(sysm[i]) > 0 and out['sys'][i].argmax().item() == sys_[i].item()),
                mae_a=err_len[0], mae_b=err_len[1], mae_c=err_len[2],
                mape_a=ape_len[0], mape_b=ape_len[1], mape_c=ape_len[2],
                mae_ang=(float(np.abs(lat_pred[i, 3:] - lat_true[i, 3:]).mean())
                         if has_lat else np.nan),
                mae_vol=(abs(vol_pred[i] - vol_true[i]) if has_lat else np.nan),
                mape_vol=(100 * abs(vol_pred[i] - vol_true[i]) / max(vol_true[i], 1e-6)
                          if has_lat else np.nan),
                pred_el=pred_set, true_el=true_set,
            ))

    d = pd.DataFrame(rows)
    res = {'n': len(d), 'el_threshold': el_threshold}
    valid = d[d['sgm'] > 0]
    res['sg_n'] = len(valid)
    res['sg_acc'] = valid['sg_ok'].mean() if len(valid) else np.nan
    res['sg_top5'] = valid['sg_top5'].mean() if len(valid) else np.nan
    valid = d[d['sysm'] > 0]
    res['sys_n'] = len(valid)
    res['sys_acc'] = valid['sys_ok'].mean() if len(valid) else np.nan
    valid = d[d['latm'] > 0]
    res['lat_n'] = len(valid)
    if len(valid):
        for key in ['mae_a', 'mae_b', 'mae_c', 'mae_ang', 'mae_vol']:
            res[key] = valid[key].mean()
        for key in ['mape_a', 'mape_b', 'mape_c', 'mape_vol']:
            res[key + '_med'] = valid[key].median()
        res['mae_a_med'] = valid['mae_a'].median()
    valid = d[d['elm'] > 0]
    res['el_n'] = len(valid)
    if len(valid):
        tp = sum(len(r.pred_el & r.true_el) for r in valid.itertuples())
        fp = sum(len(r.pred_el - r.true_el) for r in valid.itertuples())
        fn = sum(len(r.true_el - r.pred_el) for r in valid.itertuples())
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        res['el_f1_micro'] = 2 * precision * recall / max(precision + recall, 1e-9)
        res['el_exact'] = float((valid['pred_el'] == valid['true_el']).mean())
    model.train()
    return res

def safe_metric(metrics, key, default=0.0):
    value = metrics.get(key, np.nan)
    return default if value is None or not np.isfinite(value) else float(value)

def selection_score(metrics):
    # Ошибки регрессии входят с небольшим отрицательным весом, чтобы подбор не
    # улучшал классификацию ценой полного разрушения параметров решётки.
    score = (safe_metric(metrics, 'sys_acc')
             + safe_metric(metrics, 'el_f1_micro')
             - 0.05 * safe_metric(metrics, 'mae_a')
             - 0.02 * safe_metric(metrics, 'mae_ang'))
    if SOURCE_KEY in ('opxrd', 'combined'):
        score += (0.50 * safe_metric(metrics, 'sg_acc')
                  + 0.25 * safe_metric(metrics, 'sg_top5'))
    return score

In [8]:
# ---------- обучение и внутренний подбор ----------
HEAD_PREFIXES = ('head_', 'trunk', 'lam_mlp')

def make_optimizer(model, cfg):
    backbone = [p for n, p in model.named_parameters() if not n.startswith(HEAD_PREFIXES)]
    heads = [p for n, p in model.named_parameters() if n.startswith(HEAD_PREFIXES)]
    if cfg['lr_backbone'] == 0:
        for p in backbone:
            p.requires_grad_(False)
        groups = [{'params': heads, 'lr': cfg['lr_head']}]
    else:
        groups = [
            {'params': backbone, 'lr': cfg['lr_backbone']},
            {'params': heads, 'lr': cfg['lr_head']},
        ]
    return torch.optim.AdamW(groups, weight_decay=cfg['wd'])

def make_scheduler(optimizer, total_steps):
    warmup = max(1, int(total_steps * WARMUP_FRAC))
    return torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lambda step: (step / warmup if step < warmup else
                      0.5 * (1 + math.cos(math.pi * min(
                          (step - warmup) / max(total_steps - warmup, 1), 1))))
    )

def train_epochs(model, frame, cfg, epochs):
    loader = make_loader(frame, shuffle=True)
    optimizer = make_optimizer(model, cfg)
    scheduler = make_scheduler(optimizer, max(1, epochs * len(loader)))
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    for _ in range(epochs):
        model.train()
        for batch in loader:
            batch = [b.to(DEVICE, non_blocking=AMP) for b in batch]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
                total = compute_losses(model(batch[0], batch[1]), batch)
            scaler.scale(total).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
    return model

def tune_candidate(train_frame, val_frame, cfg, max_epochs):
    model = load_pretrained()
    loader = make_loader(train_frame, shuffle=True)
    val_loader = make_loader(val_frame, shuffle=False)
    optimizer = make_optimizer(model, cfg)
    scheduler = make_scheduler(optimizer, max(1, max_epochs * len(loader)))
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)

    best_score = -np.inf
    best_epoch = 1
    best_state = None
    stale = 0
    for epoch in range(1, max_epochs + 1):
        model.train()
        for batch in loader:
            batch = [b.to(DEVICE, non_blocking=AMP) for b in batch]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
                total = compute_losses(model(batch[0], batch[1]), batch)
            scaler.scale(total).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

        metrics = evaluate(model, val_loader, el_threshold=0.5)
        score = selection_score(metrics)
        if score > best_score + 1e-4:
            best_score = score
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
        if epoch >= MIN_EPOCHS and stale >= PATIENCE:
            break

    model.load_state_dict(best_state)
    best_threshold = 0.5
    best_metrics = None
    best_threshold_score = -np.inf
    for threshold in EL_THRESHOLDS:
        metrics = evaluate(model, val_loader, el_threshold=threshold)
        score = selection_score(metrics)
        if score > best_threshold_score:
            best_threshold_score = score
            best_threshold = threshold
            best_metrics = metrics
    return model, best_epoch, best_threshold, best_threshold_score, best_metrics

def inner_group_split(frame, seed):
    # Пробуем несколько seed, чтобы inner validation не оказался почти без меток.
    best = None
    best_coverage = -1
    for attempt in range(30):
        splitter = GroupShuffleSplit(n_splits=1, test_size=INNER_VAL_FRAC,
                                     random_state=seed + attempt)
        tr_idx, va_idx = next(splitter.split(frame, groups=frame['conn_key']))
        va = frame.iloc[va_idx]
        coverage = (va['lattice_a'].notna().sum()
                    + va['crystal_system'].isin(SYSTEMS).sum()
                    + va['elements'].map(len).gt(0).sum()
                    + (va['spacegroup_number'].notna().sum() if SOURCE_KEY == 'opxrd' else 0))
        if coverage > best_coverage:
            best = (tr_idx, va_idx)
            best_coverage = coverage
    return best

In [9]:
# ---------- paired nested GroupKFold ----------
def source_outer_splits(frame):
    splitter = GroupKFold(n_splits=OUTER_FOLDS)
    return list(splitter.split(frame, groups=frame['conn_key']))

def source_inner_split(frame, source_key, seed):
    # Повторяет выбор inner split из source-only ноутбуков.
    best = None
    best_coverage = -1
    for attempt in range(30):
        splitter = GroupShuffleSplit(n_splits=1, test_size=INNER_VAL_FRAC,
                                     random_state=seed + attempt)
        tr_idx, va_idx = next(splitter.split(frame, groups=frame['conn_key']))
        va = frame.iloc[va_idx]
        current = (va['lattice_a'].notna().sum()
                   + va['crystal_system'].isin(SYSTEMS).sum()
                   + va['elements'].map(len).gt(0).sum()
                   + (va['spacegroup_number'].notna().sum() if source_key == 'opxrd' else 0))
        if current > best_coverage:
            best = (tr_idx, va_idx)
            best_coverage = current
    return best

rruff_outer = source_outer_splits(ft_rruff)
opxrd_outer = source_outer_splits(ft_opxrd)
fold_ids = range(OUTER_FOLDS) if MODE == 'cv' else range(1)

zero_shot_rows = []
fold_rows = []
tuning_rows = []
t0 = time.time()

for fold0 in fold_ids:
    fold = fold0 + 1
    rr_tr_idx, rr_va_idx = rruff_outer[fold0]
    op_tr_idx, op_va_idx = opxrd_outer[fold0]
    rr_train = ft_rruff.iloc[rr_tr_idx].reset_index(drop=True)
    rr_val = ft_rruff.iloc[rr_va_idx].reset_index(drop=True)
    op_train = ft_opxrd.iloc[op_tr_idx].reset_index(drop=True)
    op_val = ft_opxrd.iloc[op_va_idx].reset_index(drop=True)
    outer_train = pd.concat([rr_train, op_train], ignore_index=True)
    outer_val = pd.concat([rr_val, op_val], ignore_index=True)

    rr_in_tr, rr_in_va = source_inner_split(rr_train, 'rruff', SEED + 100 * fold)
    op_in_tr, op_in_va = source_inner_split(op_train, 'opxrd', SEED + 100 * fold)
    inner_train = pd.concat([rr_train.iloc[rr_in_tr], op_train.iloc[op_in_tr]], ignore_index=True)
    inner_val = pd.concat([rr_train.iloc[rr_in_va], op_train.iloc[op_in_va]], ignore_index=True)

    print(f'\n===== OUTER FOLD {fold}/{len(list(fold_ids))} =====')
    print('train:', len(outer_train), '| val:', len(outer_val),
          '| val RRUFF/opXRD:', len(rr_val), len(op_val))

    zs = load_pretrained()
    for scope, frame in [('combined', outer_val), ('rruff', rr_val), ('opxrd', op_val)]:
        metrics = evaluate(zs, make_loader(frame, False), el_threshold=0.5)
        zero_shot_rows.append({'fold': fold, 'scope': scope, **metrics})
    del zs
    if AMP:
        torch.cuda.empty_cache()

    candidate_results = []
    for cfg in CANDIDATES:
        print('  tuning', cfg['name'], cfg)
        tuned_model, best_epoch, threshold, score, inner_metrics = tune_candidate(
            inner_train, inner_val, cfg, MAX_TUNE_EPOCHS)
        tuning_rows.append({
            'fold': fold, 'candidate': cfg['name'], 'best_epoch': best_epoch,
            'el_threshold': threshold, 'inner_score': score, **inner_metrics,
        })
        candidate_results.append((score, cfg, best_epoch, threshold))
        print(f"    score={score:.4f} epoch={best_epoch} threshold={threshold:.2f}")
        del tuned_model
        if AMP:
            torch.cuda.empty_cache()

    _, chosen_cfg, chosen_epoch, chosen_threshold = max(candidate_results, key=lambda x: x[0])
    print('  selected:', chosen_cfg['name'], '| epochs:', chosen_epoch,
          '| elements threshold:', chosen_threshold)

    model = load_pretrained()
    model = train_epochs(model, outer_train, chosen_cfg, chosen_epoch)
    for scope, frame in [('combined', outer_val), ('rruff', rr_val), ('opxrd', op_val)]:
        metrics = evaluate(model, make_loader(frame, False), chosen_threshold)
        fold_rows.append({
            'fold': fold, 'scope': scope, 'candidate': chosen_cfg['name'],
            'epochs': chosen_epoch, 'el_threshold': chosen_threshold, **metrics,
        })
        shown = {k: round(v, 4) for k, v in metrics.items()
                 if k in ['sys_acc', 'sg_acc', 'sg_top5', 'el_f1_micro',
                          'el_exact', 'mae_a', 'mae_ang'] and np.isfinite(v)}
        print('   ', scope, shown)
    del model
    if AMP:
        torch.cuda.empty_cache()

print(f'\nelapsed: {(time.time() - t0) / 60:.1f} min')


===== OUTER FOLD 1/5 =====
train: 2779 | val: 696 | val RRUFF/opXRD: 272 424
  tuning head_only {'name': 'head_only', 'lr_backbone': 0.0, 'lr_head': 8e-05, 'wd': 0.0001}


d:\Dev\envs\study\lib\site-packages\torch\optim\lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


    score=1.5505 epoch=15 threshold=0.30
  tuning conservative {'name': 'conservative', 'lr_backbone': 3e-06, 'lr_head': 3e-05, 'wd': 0.0001}


d:\Dev\envs\study\lib\site-packages\torch\optim\lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


    score=1.4415 epoch=28 threshold=0.20
  tuning balanced {'name': 'balanced', 'lr_backbone': 8e-06, 'lr_head': 5e-05, 'wd': 0.0001}


...

    score=0.9011 epoch=27 threshold=0.30
  tuning stronger {'name': 'stronger', 'lr_backbone': 1.5e-05, 'lr_head': 0.0001, 'wd': 0.0003}


d:\Dev\envs\study\lib\site-packages\torch\optim\lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


    score=0.9728 epoch=22 threshold=0.30
  selected: head_only | epochs: 28 | elements threshold: 0.3


d:\Dev\envs\study\lib\site-packages\torch\optim\lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


    combined {'sg_acc': np.float64(0.8919), 'sg_top5': np.float64(1.0), 'sys_acc': np.float64(0.6711), 'mae_a': np.float64(2.2811), 'mae_ang': np.float64(4.4178), 'el_f1_micro': 0.643, 'el_exact': 0.1631}
    rruff {'sys_acc': np.float64(0.6332), 'mae_a': np.float64(2.2381), 'mae_ang': np.float64(5.0365), 'el_f1_micro': 0.5859, 'el_exact': 0.0148}
    opxrd {'sg_acc': np.float64(0.8919), 'sg_top5': np.float64(1.0), 'sys_acc': np.float64(0.707), 'mae_a': np.float64(2.3219), 'mae_ang': np.float64(3.8309), 'el_f1_micro': 0.8188, 'el_exact': 0.4276}

elapsed: 40.7 min


In [10]:
# ---------- сводка и прямое сравнение со специалистами ----------
tuning_df = pd.DataFrame(tuning_rows)
fold_df = pd.DataFrame(fold_rows)
zero_df = pd.DataFrame(zero_shot_rows)

metric_keys = [
    'sys_acc', 'sg_acc', 'sg_top5', 'el_f1_micro', 'el_exact',
    'mae_a', 'mae_a_med', 'mape_a_med', 'mae_b', 'mape_b_med',
    'mae_c', 'mape_c_med', 'mae_ang', 'mae_vol', 'mape_vol_med'
]
summary_rows = []
for scope in ['combined', 'rruff', 'opxrd']:
    before_scope = zero_df[zero_df['scope'] == scope]
    after_scope = fold_df[fold_df['scope'] == scope]
    for key in metric_keys:
        before = before_scope[key].dropna() if key in before_scope else pd.Series(dtype=float)
        after = after_scope[key].dropna() if key in after_scope else pd.Series(dtype=float)
        summary_rows.append({
            'scope': scope, 'metric': key,
            'zero_shot': before.mean() if len(before) else np.nan,
            'after_combined_ft': after.mean() if len(after) else np.nan,
            'std': after.std(ddof=0) if len(after) > 1 else np.nan,
            'folds': len(after),
        })
summary_df = pd.DataFrame(summary_rows)

print('\n===== COMBINED CONTROL SUMMARY =====')
for scope in ['combined', 'rruff', 'opxrd']:
    print('\n--', scope, '--')
    print(summary_df[summary_df['scope'] == scope].to_string(
        index=False, float_format=lambda x: f'{x:.4f}'))

# Попарная проверка: одинаковый source и одинаковый номер outer fold.
comparison_rows = []
specialist_files = {
    'rruff': OUT / 'ft_rruff_only_fold_metrics.csv',
    'opxrd': OUT / 'ft_opxrd_only_fold_metrics.csv',
}
for scope, specialist_path in specialist_files.items():
    specialist = pd.read_csv(specialist_path)
    control = fold_df[fold_df['scope'] == scope]
    paired = control.merge(specialist, on='fold', suffixes=('_combined', '_specialist'))
    for key in metric_keys:
        c_col, s_col = key + '_combined', key + '_specialist'
        if c_col in paired and s_col in paired:
            valid = paired[[c_col, s_col]].dropna()
            if len(valid):
                delta = valid[c_col] - valid[s_col]
                comparison_rows.append({
                    'scope': scope, 'metric': key,
                    'combined_ft': valid[c_col].mean(),
                    'specialist_ft': valid[s_col].mean(),
                    'difference_combined_minus_specialist': delta.mean(),
                    'std_paired_difference': delta.std(ddof=0) if len(delta) > 1 else np.nan,
                    'folds': len(valid),
                })

comparison_df = pd.DataFrame(comparison_rows)
print('\n===== PAIRED: COMBINED MINUS SPECIALIST =====')
print(comparison_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print('Для accuracy/F1 положительная разница лучше combined; для ошибок отрицательная лучше combined.')

tuning_df.to_csv(OUT / f'{OUTPUT_PREFIX}_tuning.csv', index=False)
fold_df.to_csv(OUT / f'{OUTPUT_PREFIX}_fold_metrics.csv', index=False)
summary_df.to_csv(OUT / f'{OUTPUT_PREFIX}_cv_summary.csv', index=False)
comparison_df.to_csv(OUT / f'{OUTPUT_PREFIX}_vs_specialists.csv', index=False)
print('saved outputs with prefix:', OUTPUT_PREFIX)


===== COMBINED CONTROL SUMMARY =====

-- combined --
   scope       metric  zero_shot  after_combined_ft      std  folds
combined      sys_acc     0.3440             0.6484   0.0275      5
combined       sg_acc     0.0656             0.5771   0.2219      5
combined      sg_top5     0.1370             0.7459   0.2468      5
combined  el_f1_micro     0.1797             0.5730   0.0478      5
combined     el_exact     0.0019             0.1970   0.0508      5
combined        mae_a     3.4650             2.5483   0.3793      5
combined    mae_a_med     2.1468             1.4632   0.1106      5
combined   mape_a_med    34.2713            18.6355   1.1998      5
combined        mae_b     3.4637             2.6050   0.1837      5
combined   mape_b_med    31.5792            17.2826   2.0146      5
combined        mae_c     4.2858             3.3735   0.2514      5
combined   mape_c_med    37.6693            21.0036   0.5004      5
combined      mae_ang     6.0409             4.5224   0.2236  

In [11]:
# ---------- финальный combined checkpoint ----------
if MODE == 'cv':
    candidate_scores = (tuning_df.groupby('candidate', as_index=False)['inner_score']
                        .mean().sort_values('inner_score', ascending=False))
    final_name = candidate_scores.iloc[0]['candidate']
    final_cfg = next(c for c in CANDIDATES if c['name'] == final_name)
    chosen = tuning_df[tuning_df['candidate'] == final_name]
    final_epochs = int(np.clip(np.median(chosen['best_epoch']), MIN_EPOCHS, MAX_TUNE_EPOCHS))
    final_threshold = float(np.median(chosen['el_threshold']))

    print('final config:', final_cfg)
    print('final epochs:', final_epochs, '| elements threshold:', final_threshold)
    final_model = load_pretrained()
    final_model = train_epochs(final_model, ft.reset_index(drop=True), final_cfg, final_epochs)
    checkpoint_path = CKPT_DIR / f'{OUTPUT_PREFIX}_final.pt'
    torch.save(final_model.state_dict(), checkpoint_path)

    metadata = {
        'source': SOURCE_NAME,
        'pretrain_checkpoint': str(CKPT_PRE),
        'config': final_cfg,
        'epochs': final_epochs,
        'elements_threshold': final_threshold,
        'nested_cv': True,
        'paired_source_folds': True,
        'synthetic_replay': False,
        'rows': len(ft),
    }
    (OUT / f'{OUTPUT_PREFIX}_final_config.json').write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
    print('saved:', checkpoint_path)
else:
    print('MODE=quick: final checkpoint is not created')

final config: {'name': 'stronger', 'lr_backbone': 1.5e-05, 'lr_head': 0.0001, 'wd': 0.0003}
final epochs: 15 | elements threshold: 0.3


d:\Dev\envs\study\lib\site-packages\torch\optim\lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


saved: D:\Users\user\Desktop\DS_XRD_project\checkpoints\ft_combined_no_replay_control_final.pt


## Как читать контроль

Главный файл после выполнения — outputs/ft_combined_no_replay_control_vs_specialists.csv.

Для accuracy и F1 положительная разница combined minus specialist означает преимущество смешанного FT. Для MAE и других ошибок преимущество combined выражается отрицательной разницей.

Если специалист стабильно лучше на одинаковых folds, смешивание действительно мешает этой голове. Если combined не хуже или лучше, различие старых результатов было связано не только со смешиванием, а также с replay, параметрами обучения или составом validation.